# Step 1: Create Data Lake Structure
We will create the directory structure `data_lake/bronze` and `data_lake/silver/preprocessed_sales` as specified in the instructions and copy our raw file to the bronze layer.

In [21]:
import os
import shutil

# Create folders
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Find the correct raw dataset and copy it to bronze
source_path = "/content/sales (3).csv"
bronze_path = "data_lake/bronze/sales.csv"

# Verify source file exists before copying
if os.path.exists(source_path):
    shutil.copy(source_path, bronze_path)
    print("File successfully copied to Bronze!")
else:
    # Fallback to look for any csv file in /content in case of minor name variations
    csv_files = [f for f in os.listdir('/content') if f.endswith('.csv')]
    if csv_files:
        fallback_source = os.path.join('/content', csv_files[0])
        shutil.copy(fallback_source, bronze_path)
        print(f"File copied using fallback: {fallback_source}")
    else:
        raise FileNotFoundError("Please make sure 'sales (3).csv' is uploaded inside the /content folder.")

print("Bronze directory content:", os.listdir("data_lake/bronze"))

File successfully copied to Bronze!
Bronze directory content: ['sales.csv']


In [22]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Create/Retrieve Spark session
spark = SparkSession.builder \
    .appName("Spark Data Lake Preprocessing") \
    .getOrCreate()

# 1. Load dataset from Bronze
raw_df = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)
print("Total Raw Rows:", raw_df.count())

# Enable legacy time parser globally
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# 2. Preprocess and Clean
cleaned_df = raw_df.dropDuplicates()

# Standardize lowercase column naming to safely identify attributes
# Clean string fields
for col in cleaned_df.columns:
    dtype = cleaned_df.schema[col].dataType
    from pyspark.sql.types import StringType
    if isinstance(dtype, StringType):
        cleaned_df = cleaned_df.withColumn(col, F.trim(F.col(col)))

# Numeric ranges standardization (handling quantity, unit_price, discount_percent)
for col in cleaned_df.columns:
    col_lower = col.lower()
    if "quantity" in col_lower:
        cleaned_df = cleaned_df.withColumn(col, F.coalesce(F.col(col).cast("integer"), F.lit(0)))
        cleaned_df = cleaned_df.withColumn(col, F.when(F.col(col) < 0, 0).otherwise(F.col(col)))
    elif "price" in col_lower:
        cleaned_df = cleaned_df.withColumn(col, F.coalesce(F.col(col).cast("double"), F.lit(0.0)))
        cleaned_df = cleaned_df.withColumn(col, F.when(F.col(col) < 0, 0.0).otherwise(F.col(col)))
    elif "discount" in col_lower:
        cleaned_df = cleaned_df.withColumn(col, F.coalesce(F.col(col).cast("double"), F.lit(0.0)))
        cleaned_df = cleaned_df.withColumn(col, F.when(F.col(col) < 0, 0.0).otherwise(F.col(col)))
        cleaned_df = cleaned_df.withColumn(col, F.when(F.col(col) > 100, 100.0).otherwise(F.col(col)))

# Date standardizations
for col in cleaned_df.columns:
    if "date" in col.lower():
        cleaned_df = cleaned_df.withColumn(col, F.coalesce(
            F.to_date(F.expr(f"try_to_timestamp({col}, 'yyyy-MM-dd')")),
            F.to_date(F.expr(f"try_to_timestamp({col}, 'MM/dd/yyyy')")),
            F.to_date(F.expr(f"try_to_timestamp({col}, 'dd-MM-yyyy')"))
        ))

# 3. Save to Silver Layer
output_silver_dir = "data_lake/silver/preprocessed_sales"
cleaned_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_silver_dir)

print("Preprocessed Silver Rows Count:", cleaned_df.count())
print("Silver folder contents:", os.listdir(output_silver_dir))

Total Raw Rows: 1000
Preprocessed Silver Rows Count: 990
Silver folder contents: ['_SUCCESS', '.part-00000-a5064d6e-0392-492e-8ad8-2427d22813f9-c000.csv.crc', 'part-00000-a5064d6e-0392-492e-8ad8-2427d22813f9-c000.csv', '._SUCCESS.crc']


In [23]:
# Cell cleanup: This block was converted to standard Python comments to avoid any execution errors.

# Step 2: Install PySpark and Initialize Spark Session
We install `pyspark` and instantiate a new SparkSession to load and inspect the dataset.

In [24]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Create spark session
spark = SparkSession.builder \
    .appName("Spark Data Lake Preprocessing") \
    .getOrCreate()

print("Spark Session Created:", spark)

Spark Session Created: <pyspark.sql.session.SparkSession object at 0x7a7074179be0>


# Step 3: Load and Inspect Raw Bronze Data
We read the `sales.csv` from the Bronze layer using PySpark, examine the schema, and retrieve initial statistics.

In [25]:
# Load dataset
raw_df = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)

# Display metadata
print("--- Raw Dataset Schema ---")
raw_df.printSchema()

print(f"Total Row Count: {raw_df.count()}")

print("--- First 5 Rows ---")
raw_df.show(5)

--- Raw Dataset Schema ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Total Row Count: 1000
--- First 5 Rows ---
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+-

# Step 4: Inspect Discrepancies and Issues via Spark
Let's programmatically find duplicates, null/missing counts, and invalid numbers or dates.

In [26]:
# 1. Duplicate rows
total_rows = raw_df.count()
distinct_rows = raw_df.distinct().count()
print(f"Total Duplicate Rows: {total_rows - distinct_rows}")

# 2. Null values in each column (handling type checks safely)
print("\n--- Null/Missing Counts per Column ---")
from pyspark.sql.types import StringType

null_checks = []
for col_name in raw_df.columns:
    dtype = raw_df.schema[col_name].dataType
    # Only check for empty string if the column is of type String
    if isinstance(dtype, StringType):
        cond = F.col(col_name).isNull() | (F.col(col_name) == 'null') | (F.col(col_name) == '')
    else:
        cond = F.col(col_name).isNull()
    null_checks.append(F.count(F.when(cond, col_name)).alias(col_name))

raw_df.select(null_checks).show()

# 3. Numeric Columns Outliers/Issues using lowercase column names
print("--- Summary Statistics of Numeric Fields ---")
raw_df.select("quantity", "unit_price", "discount_percent").summary("count", "min", "max").show()

Total Duplicate Rows: 10

--- Null/Missing Counts per Column ---
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

--- Summary Statistics of Numeric Fields ---
+-------+--------+----------+----------------+
|summary|quantity|unit_price|discount_percent|
+-------+--------+----------+----------------+
|  count|    1000|      1000|            10

# Step 5: Preprocessing for Silver Layer
Now we resolve issues step-by-step:
- Remove exact duplicates
- Clean string columns (trim, handle cases)
- Handle missing/invalid numerical columns
- Standardize dates

In [27]:
# Enable legacy parser policy globally on Spark before transforming dates so it tolerates invalid calendar inputs
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# 1. Remove exact duplicate rows
cleaned_df = raw_df.dropDuplicates()

# 2. String Fields cleaning: trim space, standardize case for text columns if present
string_cols = [item[0] for item in cleaned_df.dtypes if item[1] == 'string']
for col in string_cols:
    cleaned_df = cleaned_df.withColumn(col, F.trim(F.col(col)))

# 3. Numeric conversions and default checks (applying lowercase column names)
if "quantity" in cleaned_df.columns:
    cleaned_df = cleaned_df.withColumn("quantity", F.coalesce(F.col("quantity").cast("integer"), F.lit(0)))
    cleaned_df = cleaned_df.withColumn("quantity", F.when(F.col("quantity") < 0, 0).otherwise(F.col("quantity")))

if "unit_price" in cleaned_df.columns:
    cleaned_df = cleaned_df.withColumn("unit_price", F.coalesce(F.col("unit_price").cast("double"), F.lit(0.0)))
    cleaned_df = cleaned_df.withColumn("unit_price", F.when(F.col("unit_price") < 0, 0.0).otherwise(F.col("unit_price")))

if "discount_percent" in cleaned_df.columns:
    cleaned_df = cleaned_df.withColumn("discount_percent", F.coalesce(F.col("discount_percent").cast("double"), F.lit(0.0)))
    cleaned_df = cleaned_df.withColumn("discount_percent", F.when(F.col("discount_percent") < 0, 0.0).otherwise(F.col("discount_percent")))
    cleaned_df = cleaned_df.withColumn("discount_percent", F.when(F.col("discount_percent") > 100, 100.0).otherwise(F.col("discount_percent")))

# 4. Standardizing Date Formats safely using try_to_timestamp SQL expression (returns NULL for impossible dates like 31/02/2026 instead of throwing errors)
if "order_date" in cleaned_df.columns:
    cleaned_df = cleaned_df.withColumn("order_date", F.coalesce(
        F.to_date(F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')")),
        F.to_date(F.expr("try_to_timestamp(order_date, 'MM/dd/yyyy')")),
        F.to_date(F.expr("try_to_timestamp(order_date, 'dd-MM-yyyy')"))
    ))

print("Cleaned Data Overview:")
cleaned_df.show(5)

Cleaned Data Overview:
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|       2|     670.0|            30.0|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|       4|     670.0|             0.0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  100506|      C0228|     Diya Tripathi|Competitive Exam ...|    Books|       6|     610.0|            12.

# Step 6: Save and Verify Silver Layer Data
We save the cleaned and preprocessed dataset into the `silver/preprocessed_sales` directory as requested, then read it back to verify success.

In [28]:
output_silver_dir = "data_lake/silver/preprocessed_sales"

# Save the DataFrame to silver layer (coalesce to 1 file for submission convenience)
cleaned_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_silver_dir)

# Verification
print(f"Rows Before Preprocessing: {raw_df.count()}")
print(f"Rows After Preprocessing: {cleaned_df.count()}")

# Read back the saved Silver data
silver_df = spark.read.csv(output_silver_dir, header=True, inferSchema=True)
print("\n--- Loaded Silver Layer Sample ---")
silver_df.show(5)

Rows Before Preprocessing: 1000
Rows After Preprocessing: 990

--- Loaded Silver Layer Sample ---
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|       2|     670.0|            30.0|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|       4|     670.0|             0.0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  100506|      C0228|     Diya 

### Step 7: Build Gold Layer Analytics and KPI Aggregations
In the Gold Layer, we transform our cleaned Silver-tier data into aggregated business metrics. We will create a summary table that calculates key metrics by product category:
- Total revenue generated (taking discounts into account)
- Total quantity sold
- Average discount percentage applied
- Total unique transactions

In [29]:
import os
from pyspark.sql import functions as F

# Ensure Spark session is active
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Ensure silver_df is loaded
output_silver_dir = "data_lake/silver/preprocessed_sales"
if 'silver_df' not in globals():
    print("Loading silver_df from:", output_silver_dir)
    silver_df = spark.read.csv(output_silver_dir, header=True, inferSchema=True)

# 1. Calculate transaction-level net sales: quantity * unit_price * (1 - discount_percent/100)
silver_with_revenue = silver_df.withColumn(
    "net_sales",
    F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - (F.col("discount_percent") / 100.0)), 2)
)

# 2. Aggregate sales metrics at a product category level for the Gold Layer
gold_category_summary = silver_with_revenue.groupBy("category").agg(
    F.sum("quantity").alias("total_units_sold"),
    F.round(F.sum("net_sales"), 2).alias("total_revenue"),
    F.round(F.mean("discount_percent"), 2).alias("average_discount_percent"),
    F.count("order_id").alias("total_transactions")
).orderBy(F.desc("total_revenue"))

# 3. Create Gold Layer directory and save the analytics summary
output_gold_dir = "data_lake/gold/category_sales_summary"
os.makedirs("data_lake/gold", exist_ok=True)

gold_category_summary.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_gold_dir)

print("--- Gold Layer Category Sales Summary Sample ---")
gold_category_summary.show()
print("Gold layer contents successfully saved to:", output_gold_dir)

--- Gold Layer Category Sales Summary Sample ---
+---------------+----------------+-------------+------------------------+------------------+
|       category|total_units_sold|total_revenue|average_discount_percent|total_transactions|
+---------------+----------------+-------------+------------------------+------------------+
|      Furniture|             704|    8639378.5|                   21.13|               129|
|    Electronics|             549|    7270942.7|                    17.7|               103|
|Home Appliances|             596|    2761348.4|                   23.62|               113|
|         Sports|             737|    1651746.8|                    19.8|               136|
|       Clothing|             557|    1341868.6|                   18.51|               107|
|         Beauty|             755|     801224.8|                    20.4|               135|
|          Books|             709|     415046.2|                   18.84|               127|
|        Grocery|    